In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

project_root = Path.cwd().parent

recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)


recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)



In [3]:
import pandas as pd

products = pd.read_csv(
    recommendation_dir/"products_processed_popularity.csv",
    low_memory=False
)

reviews = pd.read_csv(
    recommendation_dir/"reviews_processed_popularity.csv",
    low_memory=False
)

In [4]:
print("Products:", products.shape)
print("Reviews:", reviews.shape)

Products: (8494, 47)
Reviews: (1088891, 24)


In [5]:
reviews["author_id"].nunique()

503216

In [6]:
reviews["product_id"].nunique()

2351

In [7]:
reviews.groupby(
    "author_id"
).size().describe()

count    503216.000000
mean          2.163864
std           3.392856
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         292.000000
dtype: float64

In [8]:
reviews.groupby(
    "product_id"
).size().describe()

count     2351.000000
mean       463.160783
std        881.165224
min          1.000000
25%         37.000000
50%        164.000000
75%        488.000000
max      15853.000000
dtype: float64

implicit feedback score

In [9]:
reviews["rating_score"] = (
    reviews["rating"] / 5
)

In [10]:
reviews["recommendation_score"] = (
    reviews["is_recommended"]
    .fillna(0)
)

In [11]:
reviews["helpfulness_score"] = (
    reviews["helpfulness"]
    .fillna(0)
)

In [12]:
reviews["interaction_score"] = (
    0.5 * reviews["rating_score"]
    +
    0.3 * reviews["recommendation_score"]
    +
    0.2 * reviews["helpfulness_score"]
)

In [13]:
reviews[
    [
        "rating",
        "is_recommended",
        "helpfulness",
        "interaction_score"
    ]
].head()

,rating,is_recommended,helpfulness,interaction_score
0,1,0.0,0.5,0.2
1,5,1.0,1.0,1.0
2,5,1.0,1.0,1.0
3,5,1.0,1.0,1.0
4,5,1.0,0.5,0.9


In [14]:
interaction_df = reviews[
    [
        "author_id",
        "product_id",
        "interaction_score"
    ]
].copy()

In [15]:
interaction_df.head()

,author_id,product_id,interaction_score
0,538863,P420652,0.2
1,549704,P218700,1.0
2,557770,P232903,1.0
3,562130,P381030,1.0
4,582399,P420652,0.9


In [16]:
interaction_df.duplicated(
    [
        "author_id",
        "product_id"
    ]
).sum()

np.int64(5)

In [17]:
interaction_df = (
    interaction_df
    .groupby(
        [
            "author_id",
            "product_id"
        ]
    )
    ["interaction_score"]
    .mean()
    .reset_index()
)

In [18]:
interaction_df.duplicated(
    [
        "author_id",
        "product_id"
    ]
).sum()

np.int64(0)

In [19]:
num_users = interaction_df["author_id"].nunique()

num_products = interaction_df["product_id"].nunique()

num_interactions = len(interaction_df)


print("Users:", num_users)
print("Products:", num_products)
print("Interactions:", num_interactions)

Users: 503216
Products: 2351
Interactions: 1088886


In [20]:
user_activity = (
    interaction_df
    .groupby("author_id")
    .size()
    .describe()
)

product_activity = (
    interaction_df
    .groupby("product_id")
    .size()
    .describe()
)

print(user_activity)
print(product_activity)

count    503216.000000
mean          2.163854
std           3.392853
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         292.000000
dtype: float64
count     2351.000000
mean       463.158656
std        881.160504
min          1.000000
25%         37.000000
50%        164.000000
75%        488.000000
max      15853.000000
dtype: float64


In [21]:
MIN_USER_INTERACTIONS = 5
MIN_PRODUCT_INTERACTIONS = 20

In [22]:
active_users = (
    interaction_df["author_id"]
    .value_counts()
)

active_users = active_users[
    active_users >= MIN_USER_INTERACTIONS
].index

In [23]:
active_products = (
    interaction_df["product_id"]
    .value_counts()
)

active_products = active_products[
    active_products >= MIN_PRODUCT_INTERACTIONS
].index

In [24]:
cf_df = interaction_df[
    interaction_df["author_id"].isin(active_users)
    &
    interaction_df["product_id"].isin(active_products)
].copy()

In [25]:
print(cf_df.shape)

print(
    cf_df["author_id"].nunique()
)

print(
    cf_df["product_id"].nunique()
)

(369815, 3)
40433
1931


In [26]:
cf_df.head()

,author_id,product_id,interaction_score
7,10000117144,P394639,0.9
8,10000117144,P4016,0.9
9,10000117144,P427415,0.9
10,10000117144,P428250,1.0
11,10000117144,P440651,0.9


In [27]:
from sklearn.preprocessing import LabelEncoder

In [28]:
user_encoder = LabelEncoder()

cf_df["user_idx"] = user_encoder.fit_transform(
    cf_df["author_id"]
)

In [29]:
product_encoder = LabelEncoder()

cf_df["product_idx"] = product_encoder.fit_transform(
    cf_df["product_id"]
)

In [30]:
cf_df.head()



,author_id,product_id,interaction_score,user_idx,product_idx
7,10000117144,P394639,0.9,0,193
8,10000117144,P4016,0.9,0,230
9,10000117144,P427415,0.9,0,455
10,10000117144,P428250,1.0,0,471
11,10000117144,P440651,0.9,0,638


In [31]:
print(
    cf_df["user_idx"].nunique()
)

print(
    cf_df["product_idx"].nunique()
)

40433
1931


In [32]:
from scipy.sparse import csr_matrix

In [33]:
user_item_matrix = csr_matrix(
    (
        cf_df["interaction_score"],
        (
            cf_df["user_idx"],
            cf_df["product_idx"]
        )
    )
)

In [34]:
user_item_matrix.shape

(40433, 1931)

In [35]:
user_item_matrix.nnz

369815

In [36]:
item_user_matrix = user_item_matrix.T

In [37]:
item_user_matrix.shape

(1931, 40433)

In [38]:
from sklearn.neighbors import NearestNeighbors

In [39]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=10
)

In [40]:
knn_model.fit(
    item_user_matrix
)

,n_neighbors,10
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [41]:
product_idx = 100

In [42]:
distances, indices = knn_model.kneighbors(
    item_user_matrix[product_idx],
    n_neighbors=10
)

In [43]:
indices

array([[ 100,  416,  347,  341,  711,  503, 1548,  907, 1187,  112]])

In [44]:
distances

array([[0.        , 0.91808737, 0.92728259, 0.93463811, 0.93673153,
        0.94270749, 0.94315875, 0.94501808, 0.94589009, 0.94629888]])

In [45]:
similar_product_ids = product_encoder.inverse_transform(
    indices.flatten()
)

In [46]:
similar_product_ids

array(['P375864', 'P423145', 'P417323', 'P417115', 'P443358', 'P430813',
       'P480444', 'P454939', 'P468709', 'P378219'], dtype=object)

In [47]:
product_lookup = (
    products[
        [
            "product_id",
            "product_name",
            "brand_name"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

product_lookup["product_idx"] = range(
    len(product_lookup)
)

product_lookup.head()

,product_id,product_name,brand_name,product_idx
0,P473671,Fragrance Discovery Set,19-69,0
1,P473668,La Habana Eau de Parfum,19-69,1
2,P473662,Rainbow Bar Eau de Parfum,19-69,2
3,P473660,Kasbah Eau de Parfum,19-69,3
4,P473658,Purple Haze Eau de Parfum,19-69,4


In [48]:
product_lookup[
    product_lookup["product_idx"]==100
]

,product_id,product_name,brand_name,product_idx
100,P392945,GENIUS Ultimate Anti-Aging Vitamin C+ Serum,Algenist,100


In [49]:
similar_products = product_lookup[
    product_lookup["product_idx"].isin(
        indices.flatten()
    )
]

similar_products

,product_id,product_name,brand_name,product_idx
100,P392945,GENIUS Ultimate Anti-Aging Vitamin C+ Serum,Algenist,100
112,P404168,POWER Advanced Wrinkle Fighter Moisturizer,Algenist,112
341,P470001,False Top Lashes,Anastasia Beverly Hills,341
347,P416785,Power Fabric + Longwear High Cover Foundation ...,Armani Beauty,347
416,P500420,The Light Cream,Augustinus Bader,416
503,P504160,BESTIES GLITTER Blend & Cleanse Starter Set,beautyblender,503
711,P501699,"802 Peony, Lotus and Bamboo Eau de Parfum",Bon Parfumeur,711
907,P454344,Hairdresser's Invisible Oil Soft Texture Finis...,Bumble and bumble,907
1187,P12542,PLATINUM ÉGOÏSTE Eau de Toilette,CHANEL,1187
1548,P474942,Refresh in 5,CLINIQUE,1548


In [50]:
user_counts = interaction_df.groupby(
    "author_id"
).size()


product_counts = interaction_df.groupby(
    "product_id"
).size()


user_counts.describe()

count    503216.000000
mean          2.163854
std           3.392853
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         292.000000
dtype: float64

In [51]:
product_counts.describe()

count     2351.000000
mean       463.158656
std        881.160504
min          1.000000
25%         37.000000
50%        164.000000
75%        488.000000
max      15853.000000
dtype: float64

In [52]:
active_users = user_counts[
    user_counts >= 3
].index


popular_products = product_counts[
    product_counts >= 10
].index

In [53]:
cf_data = interaction_df[
    interaction_df["author_id"].isin(active_users)
    &
    interaction_df["product_id"].isin(popular_products)
].copy()

In [54]:
cf_data.shape

(585824, 3)

In [55]:
cf_data["author_id"].nunique()


104861

In [56]:
cf_data["product_id"].nunique()

2122

In [57]:
from sklearn.preprocessing import LabelEncoder


user_encoder = LabelEncoder()
product_encoder = LabelEncoder()


cf_data["user_idx"] = user_encoder.fit_transform(
    cf_data["author_id"]
)


cf_data["product_idx"] = product_encoder.fit_transform(
    cf_data["product_id"]
)


cf_data.head()

,author_id,product_id,interaction_score,user_idx,product_idx
7,10000117144,P394639,0.9,0,199
8,10000117144,P4016,0.9,0,236
9,10000117144,P427415,0.9,0,467
10,10000117144,P428250,1.0,0,488
11,10000117144,P440651,0.9,0,673


In [58]:
num_users = cf_data["user_idx"].nunique()
num_products = cf_data["product_idx"].nunique()

print("Users:", num_users)
print("Products:", num_products)

Users: 104861
Products: 2122


In [59]:
from scipy.sparse import csr_matrix


user_product_matrix = csr_matrix(
    (
        cf_data["interaction_score"],
        (
            cf_data["user_idx"],
            cf_data["product_idx"]
        )
    ),
    shape=(
        num_users,
        num_products
    )
)

In [60]:
user_product_matrix.shape

(104861, 2122)

In [61]:
user_product_matrix.nnz

585824

In [62]:
item_user_matrix = user_product_matrix.T

item_user_matrix.shape

(2122, 104861)

In [63]:
from sklearn.neighbors import NearestNeighbors


item_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)


item_knn.fit(
    item_user_matrix
)

,n_neighbors,11
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,-1


In [64]:
distances, indices = item_knn.kneighbors(
    item_user_matrix[100],
    n_neighbors=10
)

In [65]:
product_lookup = (
    cf_data[
        [
            "product_idx",
            "product_id"
        ]
    ]
    .drop_duplicates()
    .set_index("product_idx")
)

In [66]:
product_lookup.head()

,product_id
product_idx,
199,P394639
236,P4016
467,P427415
488,P428250
673,P440651


In [67]:
products_cf = products[
    [
        "product_id",
        "product_name",
        "brand_name",
        "primary_category",
        "secondary_category"
    ]
].drop_duplicates()


products_cf.head()

,product_id,product_name,brand_name,primary_category,secondary_category
0,P473671,Fragrance Discovery Set,19-69,Fragrance,Value & Gift Sets
1,P473668,La Habana Eau de Parfum,19-69,Fragrance,Women
2,P473662,Rainbow Bar Eau de Parfum,19-69,Fragrance,Women
3,P473660,Kasbah Eau de Parfum,19-69,Fragrance,Women
4,P473658,Purple Haze Eau de Parfum,19-69,Fragrance,Women


In [68]:
product_map = (
    cf_data[
        [
            "product_idx",
            "product_id"
        ]
    ]
    .drop_duplicates()
    .merge(
        products_cf,
        on="product_id",
        how="left"
    )
)


product_map.head()

,product_idx,product_id,product_name,brand_name,primary_category,secondary_category
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size


In [69]:
def recommend_similar_products(product_id, top_n=10, threshold=0.05):

    if product_id not in product_map["product_id"].values:
        return "Product not found"


    product_idx = (
        product_map[
            product_map["product_id"] == product_id
        ]["product_idx"]
        .values[0]
    )


    distances, indices = item_knn.kneighbors(
        item_user_matrix[product_idx],
        n_neighbors=top_n+1
    )


    recommendations=[]


    for distance, idx in zip(
        distances[0][1:],
        indices[0][1:]
    ):

        similarity = 1-distance


        if similarity < threshold:
            continue


        row = product_map[
            product_map["product_idx"] == idx
        ].iloc[0]


        recommendations.append(
            {
                "product_id":row["product_id"],
                "product_name":row["product_name"],
                "brand":row["brand_name"],
                "category":row["primary_category"],
                "similarity_score":round(similarity,3)
            }
        )


    return pd.DataFrame(recommendations)

In [70]:
recommend_similar_products(
    "P392945",
    top_n=10,
    threshold=0.05
)

,product_id,product_name,brand,category,similarity_score
0,P388200,GENIUS Ultimate Anti-Aging Melting Cleanser,Algenist,Skincare,0.089
1,P384537,GENIUS Ultimate Anti-Aging Cream,Algenist,Skincare,0.063
2,P416552,POWER Recharging Night Pressed Serum,Algenist,Skincare,0.054


In [71]:
import os

os.makedirs(
    "models/recommendation",
    exist_ok=True
)

In [72]:
import pickle


with open(
    "models/recommendation/item_knn.pkl",
    "wb"
) as f:
    pickle.dump(
        item_knn,
        f
    )

In [73]:
with open(
    "models/recommendation/item_user_matrix.pkl",
    "wb"
) as f:
    pickle.dump(
        item_user_matrix,
        f
    )

In [74]:
product_map.to_csv(
    "models/recommendation/product_map.csv",
    index=False
)

In [75]:
cf_data.to_csv(
    "models/recommendation/cf_data.csv",
    index=False
)

In [78]:
reviews.to_csv(
   "models/recommendation/ reviews_processed_popularity.csv",
    index=False
)